# Learning the Logical Channel from Syndrome Information

This notebook demonstrates the full pipeline for learning logical error probabilities from circuit-level syndrome data.

**Pipeline:**
1. **Build code** — construct a CSS code (rotated surface code)
2. **Syndrome extraction** — build a noisy stim circuit, sample detector/observable outcomes, extract the DEM
3. **Benchmark** — compare sampled LEP (direct Monte Carlo decoding) vs predicted LEP (from syndrome-learned priors)

In [ ]:
import numpy as np
from sim_qec.pipeline import (
    CSSCode,
    SyndromeExtractionConfig,
    SyndromeExtractionResult,
    run_syndrome_extraction,
    benchmark_lep,
)
from sim_qec.detector_error_models.dem_sim import CircuitErrorParams

## Step 1: Build the CSS Code

We use the rotated surface code of distance $d=3$, which gives a $[[9, 1, 3]]$ code.
The `CSSCode` dataclass validates the CSS condition $H_x H_z^T = 0 \pmod{2}$ and computes $n$, $k$ (via matrix rank).

In [ ]:
d = 3
code = CSSCode.from_rotated_surface_code(d)

print(f"Code: [[{code.n}, {code.k}, {code.distance}]]")
print(f"Hx shape: {code.Hx.shape}, Hz shape: {code.Hz.shape}")
print(f"CSS condition Hx @ Hz^T = 0: {np.all((code.Hx @ code.Hz.T) % 2 == 0)}")

## Step 2: Syndrome Extraction

Configure the circuit-level noise model and run the syndrome extraction experiment.

**Error parameters** (relative weights, scaled by `physical_error_rate`):
- `p_i` — single-qubit idling error
- `p_state_p` — state preparation error
- `p_m` — measurement error
- `p_CX` — two-qubit gate (CX) depolarizing error
- `p_idling_gate` — idling error during gate operations

In [ ]:
config = SyndromeExtractionConfig(
    num_cycles=1,
    physical_error_rate=5e-4,
    shots=5_000_000,
    fault_type="DEPOLARIZE1",
    circuit_error_params=CircuitErrorParams(
        p_i=1.0,
        p_state_p=0.8,
        p_m=0.9,
        p_CX=1.0,
        p_idling_gate=0.0,
    ),
)

result = run_syndrome_extraction(code, config)

### Inspect the extraction result

In [ ]:
print(f"Detectors:    {result.detector_error_model.num_detectors}")
print(f"Observables:  {result.detector_error_model.num_observables}")
print(f"Detector array shape:    {result.det_vals.shape}")
print(f"Observable array shape:  {result.log_vals.shape}")
print(f"Check matrix shape:      {result.check_matrix.shape}")
print(f"Number of faults:        {len(result.channel_probs)}")
print(f"Syndrome expectations:   {result.syndrome_expectations}")

### Visualize the stim circuit (optional)

In [ ]:
print(result.circuit)

## Step 3: Benchmark — Sampled LEP vs Predicted LEP

Learn the fault priors from syndrome expectation values, then compare:
- **Sampled LEP**: decode all shots with BPLSD, count logical failures
- **Predicted LEP**: use syndrome-learned priors to enumerate low-weight faults and compute LEP analytically

In [ ]:
bench = benchmark_lep(result, max_order=4)

### Prior prediction quality

In [ ]:
print(f"Prior prediction L2 error: {bench.prior_l2_error:.6e}")
print(f"Predicted priors (first 10): {bench.predicted_priors[:10]}")
print(f"True priors      (first 10): {result.channel_probs[:10]}")

### LEP comparison

In [ ]:
print(f"Sampled LEP:   {bench.lep_sampled:.6e}  ({bench.lep_sampled_runtime:.3f}s)")
print(f"Predicted LEP: {bench.lep_predicted:.6e}  ({bench.lep_predicted_runtime:.3f}s)")

### Plot: predicted vs true priors

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: predicted vs true
ax = axes[0]
ax.scatter(result.channel_probs, bench.predicted_priors, s=15, alpha=0.7)
lim = max(result.channel_probs.max(), bench.predicted_priors.max()) * 1.1
ax.plot([0, lim], [0, lim], 'k--', lw=0.8, label='y = x')
ax.set_xlabel('True prior')
ax.set_ylabel('Predicted prior')
ax.set_title('Predicted vs True Fault Priors')
ax.legend()

# Bar: LEP comparison
ax = axes[1]
labels = ['Sampled', 'Predicted']
values = [bench.lep_sampled, bench.lep_predicted]
colors = ['steelblue', 'coral']
bars = ax.bar(labels, values, color=colors, width=0.5)
ax.set_ylabel('Logical Error Probability')
ax.set_title('LEP Comparison')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{val:.2e}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()